# 03.6 - Bayesian Thinking

**Phase:** 03 - Statistics & Probability
**Status:** VERIFIED
---

## What Are We Solving?
Bayesian thinking updates beliefs with evidence using Bayes' theorem. It treats parameters as random variables with distributions.

## Mental Model
Prior belief + data → posterior belief. The posterior becomes the new prior.

## Core Concepts
- prior, likelihood, posterior
- conjugate priors
- Bayesian vs frequentist interpretation
- credible intervals vs confidence intervals
- Bayesian model comparison
- regularization as prior

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Bayes' theorem: P(θ|data) ∝ P(data|θ) * P(θ)
# Beta-Binomial conjugate prior example
print("=== BETA-BINOMIAL CONJUGATE PRIOR ===")

# Prior: Beta(α=2, β=2) - weakly favors 0.5
prior_alpha, prior_beta = 2, 2
prior = stats.beta(prior_alpha, prior_beta)

# Data: 7 heads, 3 tails (10 coin flips)
heads, tails = 7, 3

# Posterior: Beta(α+heads, β+tails)
post_alpha = prior_alpha + heads
post_beta = prior_beta + tails
posterior = stats.beta(post_alpha, post_beta)

print(f"Prior: Beta({prior_alpha}, {prior_beta})")
print(f"  Mean: {prior.mean():.3f}, Mode: {(prior_alpha-1)/(prior_alpha+prior_beta-2):.3f}")
print(f"  95% CI: {prior.interval(0.95)}")

print(f"\nData: {heads} heads, {tails} tails")

print(f"\nPosterior: Beta({post_alpha}, {post_beta})")
print(f"  Mean: {posterior.mean():.3f}, Mode: {(post_alpha-1)/(post_alpha+post_beta-2):.3f}")
print(f"  95% Credible Interval: {posterior.interval(0.95)}")

=== BETA-BINOMIAL CONJUGATE PRIOR ===
Prior: Beta(2, 2)
  Mean: 0.500, Mode: 0.500
  95% CI: (np.float64(0.09429932405024612), np.float64(0.9057006759497539))

Data: 7 heads, 3 tails

Posterior: Beta(9, 5)
  Mean: 0.643, Mode: 0.667
  95% Credible Interval: (np.float64(0.38573833824929565), np.float64(0.8614206611098394))


In [2]:
# Visualize prior, likelihood, posterior
x = np.linspace(0, 1, 200)

# Likelihood (Binomial)
likelihood = stats.binom.pmf(heads, heads+tails, x) * (heads+tails+1)  # normalize for plotting

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, prior.pdf(x), 'b-', lw=2, label=f'Prior Beta({prior_alpha},{prior_beta})')
axes[0].fill_between(x, prior.pdf(x), alpha=0.3, color='blue')
axes[0].set_title('Prior Belief')
axes[0].set_xlabel('θ (probability of heads)')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].plot(x, likelihood, 'g-', lw=2, label='Likelihood')
axes[1].fill_between(x, likelihood, alpha=0.3, color='green')
axes[1].set_title('Likelihood (Data)')
axes[1].set_xlabel('θ')
axes[1].set_ylabel('Likelihood')
axes[1].legend()

axes[2].plot(x, prior.pdf(x), 'b--', lw=1, label='Prior')
axes[2].plot(x, likelihood, 'g--', lw=1, label='Likelihood')
axes[2].plot(x, posterior.pdf(x), 'r-', lw=2, label=f'Posterior Beta({post_alpha},{post_beta})')
axes[2].fill_between(x, posterior.pdf(x), alpha=0.3, color='red')
axes[2].set_title('Posterior (Updated Belief)')
axes[2].set_xlabel('θ')
axes[2].set_ylabel('Density')
axes[2].legend()

plt.tight_layout()
plt.savefig('bayesian_update.png', dpi=150, bbox_inches='tight')
print("Saved: bayesian_update.png")

Saved: bayesian_update.png


## Credible Interval vs Confidence Interval
- **Credible Interval (Bayesian)**: P(θ ∈ CI | data) = 0.95 — probability parameter is in interval
- **Confidence Interval (Frequentist)**: 95% of such intervals contain true parameter — NOT probability for this interval

In [3]:
# Compare Bayesian credible interval vs Frequentist confidence interval
print("=== CREDIBLE INTERVAL vs CONFIDENCE INTERVAL ===")

# Frequentist: Wilson score interval for binomial proportion
from statsmodels.stats.proportion import proportion_confint
ci_freq = proportion_confint(heads, heads+tails, alpha=0.05, method='wilson')
print(f"Frequentist 95% CI (Wilson): [{ci_freq[0]:.3f}, {ci_freq[1]:.3f}]")
print(f"  Interpretation: 95% of such intervals contain true p")

# Bayesian: Credible interval
ci_bayes = posterior.interval(0.95)
print(f"\nBayesian 95% Credible Interval: [{ci_bayes[0]:.3f}, {ci_bayes[1]:.3f}]")
print(f"  Interpretation: P(p ∈ [{ci_bayes[0]:.3f}, {ci_bayes[1]:.3f}] | data) = 0.95")

# With different priors
print(f"\nWith different priors:")
for prior_a, prior_b in [(1, 1), (0.5, 0.5), (10, 10), (100, 100)]:
    post = stats.beta(prior_a + heads, prior_b + tails)
    ci = post.interval(0.95)
    print(f"  Prior Beta({prior_a},{prior_b}) -> Posterior Beta({prior_a+heads},{prior_b+tails}) -> CI: [{ci[0]:.3f}, {ci[1]:.3f}]")

=== CREDIBLE INTERVAL vs CONFIDENCE INTERVAL ===


Frequentist 95% CI (Wilson): [0.397, 0.892]
  Interpretation: 95% of such intervals contain true p

Bayesian 95% Credible Interval: [0.386, 0.861]
  Interpretation: P(p ∈ [0.386, 0.861] | data) = 0.95

With different priors:
  Prior Beta(1,1) -> Posterior Beta(8,4) -> CI: [0.390, 0.891]
  Prior Beta(0.5,0.5) -> Posterior Beta(7.5,3.5) -> CI: [0.394, 0.907]
  Prior Beta(10,10) -> Posterior Beta(17,13) -> CI: [0.389, 0.736]
  Prior Beta(100,100) -> Posterior Beta(107,103) -> CI: [0.442, 0.577]


## Regularization as Prior
L2 regularization (Ridge) = Gaussian prior on coefficients
L1 regularization (Lasso) = Laplace prior on coefficients

In [4]:
# Bayesian Linear Regression (simple)
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Generate data
np.random.seed(42)
n = 20
x = np.linspace(0, 10, n)
y_true = 2 * x + 1
y = y_true + np.random.normal(0, 2, n)

X = x.reshape(-1, 1)

# OLS (no regularization = flat prior)
ols = LinearRegression()
ols.fit(X, y)

# Ridge (L2 = Gaussian prior)
ridge = Ridge(alpha=1.0)
ridge.fit(X, y)

# Stronger regularization
ridge_strong = Ridge(alpha=10.0)
ridge_strong.fit(X, y)

print(f"=== REGULARIZATION AS PRIOR ===")
print(f"True: y = 2x + 1")
print(f"OLS (flat prior): y = {ols.coef_[0]:.3f}x + {ols.intercept_:.3f}")
print(f"Ridge α=1 (weak Gaussian prior): y = {ridge.coef_[0]:.3f}x + {ridge.intercept_:.3f}")
print(f"Ridge α=10 (strong Gaussian prior): y = {ridge_strong.coef_[0]:.3f}x + {ridge_strong.intercept_:.3f}")

# Visualize
x_plot = np.linspace(0, 10, 100)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.6, label='Data')
ax.plot(x_plot, 2*x_plot + 1, 'k-', lw=2, label='True')
ax.plot(x_plot, ols.predict(x_plot.reshape(-1,1)), 'b--', label='OLS')
ax.plot(x_plot, ridge.predict(x_plot.reshape(-1,1)), 'g-', label='Ridge α=1')
ax.plot(x_plot, ridge_strong.predict(x_plot.reshape(-1,1)), 'r-', label='Ridge α=10')
ax.legend()
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Regularization as Prior')
plt.savefig('bayesian_regularization.png', dpi=150, bbox_inches='tight')
print("Saved: bayesian_regularization.png")

=== REGULARIZATION AS PRIOR ===
True: y = 2x + 1
OLS (flat prior): y = 1.622x + 2.549
Ridge α=1 (weak Gaussian prior): y = 1.613x + 2.593
Ridge α=10 (strong Gaussian prior): y = 1.538x + 2.967


Saved: bayesian_regularization.png


## Common Mistakes
- choosing an informative prior without justification
- confusing credible intervals with confidence intervals
- ignoring the prior's influence on small data
- computational complexity for complex models

## Hands-On Practice
1. **Basic**: Update a Beta prior with Binomial data.
2. **Guided**: Compare Bayesian and frequentist intervals.
3. **Independent**: Implement Bayesian linear regression with simple priors.
4. **Challenge**: Explain how L2 regularization corresponds to a Gaussian prior.

## Knowledge Check
1. What is the difference between a prior and a posterior?
2. What is a conjugate prior and why is it useful?
3. How does a credible interval differ from a confidence interval?
4. How does L2 regularization relate to a Gaussian prior?
5. When does the prior dominate the posterior?

In [5]:
# Verification
print("VERIFICATION PASSED: Phase 03.6 complete")
print("Key takeaway: Bayesian = update beliefs with data. Prior + Likelihood → Posterior. Regularization = prior.")

VERIFICATION PASSED: Phase 03.6 complete
Key takeaway: Bayesian = update beliefs with data. Prior + Likelihood → Posterior. Regularization = prior.


## Summary
- Bayes: P(θ|data) ∝ P(data|θ) × P(θ)
- Conjugate priors: Beta-Binomial, Normal-Normal, Gamma-Poisson
- Credible interval: P(θ ∈ CI | data) = 0.95
- Prior choice matters most with small data
- Ridge = Gaussian prior; Lasso = Laplace prior

## Further Experiment
- Implement MCMC with PyMC or Stan for complex models
- Bayesian model comparison with Bayes factors
- Hierarchical models for grouped data
- Variational inference for scalable Bayesian ML

## Verification Status
- **STATUS: VERIFIED**
- **EXECUTION: PASS**
- **DEPENDENCIES:** numpy, matplotlib, scipy, sklearn, statsmodels
- **OUTPUTS: PASS**
- **LAST VERIFIED: 2026-08-29**